In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PostgresAnalysis").getOrCreate()

data = [
    (1, "S01", "P100", "Drink", 2, 15.5, "2026-04-01"),
    (2, "S01", "P101", "Snack", -1, 12.0, "2026-04-01"), 
    (3, "S02", "P100", "Drink", None, 15.5, "2026-04-01"), 
    (4, "S03", "P103", "Cleaning", 5, 8.0, "2026-04-01"),
    (5, "S02", "P104", "Food", 10, 25.0, "2026-04-02")
]
columns = ["transaction_id", "store_id", "product_id", "category", "quantity", "price", "transaction_date"]
df_setup = spark.createDataFrame(data, columns)

jdbc_url = "jdbc:postgresql://postgres:5432/retail_db"
properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

df_setup.write.jdbc(url=jdbc_url, table="sales_data", mode="overwrite", properties=properties)

In [5]:
from pyspark.sql.functions import col
df_sales = spark.read.jdbc(url=jdbc_url, table="sales_data", properties=properties)
print("\n1. Dữ liệu gốc kéo từ PostgreSQL về:")
df_sales.show()
df_clean = df_sales.dropna().filter(col("quantity") > 0)
print("2. Dữ liệu sau khi làm sạch")
df_clean.show()
df_revenue = df_clean.withColumn("revenue", col("quantity") * col("price"))
print("3. Bảng dữ liệu cuối cùng kèm Doanh thu:")
df_revenue.show()


1. Dữ liệu gốc kéo từ PostgreSQL về:
+--------------+--------+----------+--------+--------+-----+----------------+
|transaction_id|store_id|product_id|category|quantity|price|transaction_date|
+--------------+--------+----------+--------+--------+-----+----------------+
|             4|     S03|      P103|Cleaning|       5|  8.0|      2026-04-01|
|             2|     S01|      P101|   Snack|      -1| 12.0|      2026-04-01|
|             1|     S01|      P100|   Drink|       2| 15.5|      2026-04-01|
|             5|     S02|      P104|    Food|      10| 25.0|      2026-04-02|
|             3|     S02|      P100|   Drink|    NULL| 15.5|      2026-04-01|
+--------------+--------+----------+--------+--------+-----+----------------+

2. Dữ liệu sau khi làm sạch (mất đi dòng số âm và dòng Null):
+--------------+--------+----------+--------+--------+-----+----------------+
|transaction_id|store_id|product_id|category|quantity|price|transaction_date|
+--------------+--------+----------+-----